# The Halting Problem

This notebook proves the undecidability of the *halting problem*.
We start with a number of definitions.

A string $t$ is a *test function with name $f$*
if $t$ has the form 
```
int f(String x) { 
    body
}
```
and, furthermore, the string $t$ can be compiled as a *Java* method, that is the evaluation of
```
eval(t)
```
does not yield an error.  Here, `eval` is a function that is provided by the Java kernel *IJava*:  It takes a string containing Java code and evaluates this code as if it were the content of a cell.  If the string defines a method, then this method can be called afterwards.

The set of all test functions is denoted as $T\!F$.  If $t \in T\!F$ and $t$ has the name $f$, then
this is written as $\mathtt{name}(t) = f$.

We provide some examples next.

In order to print the error messages of the Java compiler, we need the following two classes of the kernel IJava.

In [ ]:
import io.github.spencerpark.ijava.IJava;
import io.github.spencerpark.ijava.execution.CompilationException;

The string `t1` is a *test function* with the name `simple`.

In Java, a string that extends over several lines is written as a *text block*, which is enclosed in triple quotes `"""`.

In [ ]:
String t1 = """
    int simple(String x) {
        return 0;
    }""";

In [ ]:
t1

By executing this string, we can check whether `t1` really is a test function. 

In [ ]:
eval(t1)

Now the method `simple` is defined.

In [ ]:
simple("Hugo")

In [ ]:
eval("2 + 3")

The string `t2` is a *test function* with the name `loop`.

In [ ]:
String t2 = """
    int loop(String x) {
        while (true) {
            x = x + 1;
        }
    }""";

In [ ]:
eval(t2)

The string `t3` is not a *test function* since it is a *Python* function, but not a Java method.

In [ ]:
String t3 = """
    def hugo(x):
        return x + 1
    """;

As `t3` is not a test function, `eval(t3)` throws a `CompilationException`.  Unfortunately, this exception does not contain the error messages of the Java compiler.  Therefore, we catch the exception and print the error messages using the method `formatError` of the kernel.

In [ ]:
try {
    eval(t3);
} catch (CompilationException e) {
    System.out.println(String.join("\n", IJava.getKernelInstance().formatError(e)));
}

**Notation:**
If $n$ is the name of a Java method that takes $k$ arguments $a_1$, $\cdots$, $a_k$,
then we write 
$$n(a_1, \cdots, a_k) \leadsto r$$ 
if the evaluation of the expression $n(a_1, \cdots, a_k)$ yields the result $r$.  

If we are not
concerned with the result $r$ but only want to state that the evaluation *terminates* eventually,
then we will write
$$n(a_1, \cdots, a_k) \,\downarrow$$
and read this notation as *evaluation of $n(a_1, \cdots, a_k)$ terminates*.

If the evaluation of the expression $n(a_1, \cdots, a_k)$ does **not** terminate, this is
written as 
$$n(a_1, \cdots, a_k) \,\uparrow.$$
This notation is read as *evaluation of $n(a_1, \cdots, a_k)$ diverges*.

The *halting problem* for Java methods is the question whether there is a
Java method
```
int stops(String t, String a) {
    ...
}    
```
that takes as input a test function $t$ and a string $a$ and that satisfies the following specification:
- $t \not\in T\!F \quad\Leftrightarrow\quad \mathtt{stops}(t, a) \leadsto 2$.

  If the first argument of `stops` is not a test function, then 
  `stops(t, a) == 2`.
- $t \in T\!F \,\wedge\, \mathtt{name}(t) = n \,\wedge\, n(a)\downarrow \quad\Leftrightarrow\quad \mathtt{stops}(t, a) \leadsto 1$.

  If the first argument of `stops` is a test function with name $n$ and, furthermore,
  the evaluation of $n(a)$ terminates, then `stops(t, a) == 1`.

- $t \in T\!F \,\wedge\, \mathtt{name}(t) = n \,\wedge\, n(a)\uparrow \quad\Leftrightarrow\quad \mathtt{stops}(t, a) \leadsto 0$.

  If the first argument of `stops` is a test function with name $n$ but the evaluation of $n(a)$ 
  diverges, then `stops(t, a) == 0`.

The function `turing` is a test function with name `alan`. 
It makes use of the function `stops`.

In [ ]:
String turing = """
    int alan(String x) {
        int result = stops(x, x);
        if (result == 1) {
            while (true) {
                System.out.println("... looping ...");
            }
        }
        return result;
    }""";

Note that the method `stops` has not been defined yet.  JShell nevertheless accepts the definition of `alan`, but `alan` can only be called once `stops` has been defined.

In [ ]:
eval(turing)

As `turing` is a test function, there are only two possibilities for the call `stops(turing, turing)`.
1. Case:  `stops(turing, turing) == 0`.
   According to the specification of `stops`, then the call `alan(turing)` should diverge. 
2. Case:  `stops(turing, turing) == 1`.
   Provided the function `stops` solves the *halting problem*, then the call `alan(turing)` should terminate.
   
Let us check both cases.  First, we pretend that `stops(turing, turing) == 0`.

In [ ]:
int stops(String t, String a) {
    return 0;
}

According to the specification of the function `stops`, the call `alan(turing)` should loop.  However, it returns `0` instead.

In [ ]:
alan(turing)

Let us assume that `stops(turing, turing) == 1` next.  When `stops` is redefined, JShell automatically recompiles the method `alan`, since `alan` calls `stops`.

In [ ]:
int stops(String t, String a) {
    return 1;
}

We will see that contrary to the specification of `stops`, the call `alan(turing)` loops.

The next cell does not terminate.  In order to stop it, you have to *restart* the kernel via the menu entry **Kernel → Restart**.  *Interrupting* the kernel is not sufficient: Although the kernel accepts new cells after an interrupt, the loop keeps running in the background and keeps printing.

In [ ]:
alan(turing)

We can only conclude that there is no way that the function `stops` can satisfy its specification.
Hence the *halting problem* is unsolvable.